In [ ]:
# Authentication
import os

os.environ["HF_ENDPOINT"] = "https://ddpday.azurewebsites.net"
os.environ["HF_TOKEN"] = "" # Fill in the token shown on the screen

In [3]:
# Download the data from the Hugging Face Hub
from huggingface_hub import hf_hub_download

hexagons_BA = hf_hub_download(repo_id="Waze/data", filename="hexagons_BA.gpkg", repo_type="dataset")
neighborhoods_BA = hf_hub_download(repo_id="Waze/data", filename="neighborhoods_BA.gpkg", repo_type="dataset")
waze_data = [hf_hub_download(repo_id="Waze/data", filename=f"part.{i}.parquet", repo_type="dataset") for i in range(4)]

In [ ]:
# Import geopandas for handling geospatial data and the Wazeasy package 
import geopandas as gpd

# Load data
hexagons_BA = gpd.read_file(hexagons_BA)
neighborhoods_BA = gpd.read_file(neighborhoods_BA)
ddf = utils.load_data(waze_data, 
                      file_type = 'parquet', 
                      filter_level_5 = True, 
                      usecols = ['ts', 'geoWKT', 'uuid', 'level','length'])
ddf = ddf.repartition(npartitions=8)

In [ ]:
# Load configuration parameters from the YAML file
import yaml
with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)


# Retrieve the parameters for Buenos Aires from the configuration
timezone = config["BuenosAires"]["timezone"]["timezone_name"]
geographies = config["BuenosAires"]["geographies"]
projected_crs = config["BuenosAires"]["crs"]["projected_crs"]

In [ ]:
# Assign each traffic jam to one of the geographic areas. It determines which administrative area or hexagon each record belongs to.
ddf = utils.assign_geography_to_jams(ddf)

# Process the timestamps and convert them to the local timezone
utils.handle_time(ddf, timezone)

In [ ]:
# Generate a basic report (We specify the time period we want to analyze, and Wazeasy creates the outputs automatically)
reports.run_basic_report(ddf, start_date='2025-01-01', end_date='2025-12-31')

In [ ]:
reports.run_geog_report(ddf, geographies, start_date='2025-01-01', end_date='2025-12-31')